# LangGraph 创建多个智能体扮演明星做综艺

本 notebook 用 LangGraph 构建一档 **多智能体综艺节目**：多个不同的大模型分别扮演不同领域的明星，在主持人（glm-5.2）的引导下围绕一个话题展开讨论。

## 设计思路

1. **搜索人设**：为每位明星用 SearXNG 搜索个人信息，再由 LLM 综合成人设卡（性格 / 事迹 / 口头禅 / 观点倾向）
2. **主持人推进**：每轮由主持人抛出子问题，引导嘉宾发言
3. **嘉宾发言**：每位明星用自己的模型发言，可以表达观点、补充或反驳他人
4. **多轮讨论**：默认 2 轮，每轮所有嘉宾依次发言
5. **总结保存**：主持人总结全场，所有发言写入 `综艺讨论记录.txt`

## 模型分配

| 角色 | 明星 | 模型 | 说明 |
|------|------|------|------|
| 主持人 | — | glm-5.2 | 推进流程、总结 |
| 嘉宾1 | 雷军 | kimi-k3 | 科技企业家，小米创始人 |
| 嘉宾2 | 罗翔 | sensenova-6.8-flash-lite | 法学教授，普法网红 |
| 嘉宾3 | 周星驰 | deepseek-v4-flash | 喜剧导演、演员 |
| 嘉宾4 | 董宇辉 | deepseek-flash | 文化主播

## 图结构

```
START → setup_personas（搜索+生成人设卡）
      → host_round（主持人开场/复盘+抛子问题）
            ├─ round_num < max → celeb_speak
            └─ round_num >= max → host_close
      celeb_speak（当前明星发言）
            ├─ speaker_idx < len(celebs) → celeb_speak（本轮循环）
            └─ speaker_idx >= len(celebs) → round_advance
      round_advance（轮次+1）→ host_round
      host_close（主持人总结）→ save_transcript（写文件）→ END
```

- `setup_personas`：为每位明星搜索信息 + 生成人设卡
- `host_round`：主持人每轮开场，设置 `speaker_idx=0`
- `celeb_speak`：当前明星发言，`speaker_idx++`
- `round_advance`：轮次 +1，回到 `host_round`
- `host_close`：主持人总结
- `save_transcript`：写入 `综艺讨论记录.txt`

## 1. 准备环境

In [1]:
# 导入依赖
import os
import time
import requests
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

# 商汤 SenseNova API
API_BASE = 'https://token.sensenova.cn/v1'
API_KEY = os.getenv('SENSENOVA_API_KEY', '')

# 模型配置（kimi-k3 要求 temperature=1，其他设 0.7）
MODEL_CONFIG = {
    'glm-5.2': {'temperature': 0.5},            # 主持人
    'kimi-k3': {'temperature': 1.0},             # 雷军（强制 temperature=1）
    'sensenova-6.8-flash-lite': {'temperature': 0.7},  # 罗翔（推理模型）
    'deepseek-v4-flash': {'temperature': 0.7},   # 周星驰
    'deepseek-flash': {'temperature': 0.7},      # 董宇辉
}

# 预建模型实例池（按 model_id 取实例，避免重复建连）
llm_pool = {}
for mid, cfg in MODEL_CONFIG.items():
    llm_pool[mid] = ChatOpenAI(
        model=mid, base_url=API_BASE, api_key=API_KEY, timeout=90, **cfg
    )
print('模型池初始化完成:', list(llm_pool.keys()))

# 明星配置
CELEBRITIES = [
    {'name': '雷军', 'model': 'kimi-k3', 'role': '科技企业家，小米创始人'},
    {'name': '罗翔', 'model': 'sensenova-6.8-flash-lite', 'role': '法学教授，普法网红'},
    {'name': '周星驰', 'model': 'deepseek-v4-flash', 'role': '喜剧导演、演员'},
    {'name': '董宇辉', 'model': 'deepseek-flash', 'role': '文化主播，前新东方英语老师'},
]
HOST_MODEL = 'glm-5.2'
TOPIC = 'AI 是否会取代人类的工作'
MAX_ROUNDS = 2

# 输出文件路径（与本 notebook 同目录）
BASE_DIR = r's:\\PythonAILearningCourse\\06-langgraph-multi-agent'
TRANSCRIPT_FILE = os.path.join(BASE_DIR, '综艺讨论记录.txt')
print('讨论记录文件:', TRANSCRIPT_FILE)

# LLM 调用封装：加延迟防限流，限流时等60s重试（TPM按分钟重置），空返回时兜底 reasoning_content
def call_llm(model_id, prompt):
    max_retries = 5
    for attempt in range(max_retries + 1):
        time.sleep(5)  # 防止 TPM/RPM 限流
        llm = llm_pool[model_id]
        try:
            resp = llm.invoke(prompt)
            content = resp.content
            if not content or not content.strip():
                # 推理模型可能把内容放在 reasoning_content 里
                content = resp.additional_kwargs.get('reasoning_content', '')
            return content.strip()
        except Exception as e:
            if '429' in str(e) or 'RateLimit' in type(e).__name__:
                if attempt < max_retries:
                    print(f'  [rate_limit] 等待60s后重试({attempt+1}/{max_retries})...', flush=True)
                    time.sleep(60)
                    continue
            raise
    raise RuntimeError(model_id + '调用失败')

模型池初始化完成: ['glm-5.2', 'kimi-k3', 'sensenova-6.8-flash-lite', 'deepseek-v4-flash', 'deepseek-flash']
讨论记录文件: s:\\PythonAILearningCourse\\06-langgraph-multi-agent\综艺讨论记录.txt


## 2. 搜索工具（本地 SearXNG）

In [2]:
# 本地搜索引擎（SearXNG，端口 6688）
SEARCH_URL = 'http://localhost:6688/search'

def search_web(query, top_n=5):
    # 国内网络下默认引擎几乎全废，必须显式指定 engines=bilibili,sogou
    try:
        resp = requests.get(SEARCH_URL, params={
            'q': query, 'format': 'json', 'engines': 'bilibili,sogou'
        }, timeout=20)
        data = resp.json()
        results = data.get('results', [])[:top_n]
        if not results:
            return f'搜索{query}无结果'
        lines = []
        for i, r in enumerate(results, 1):
            title = r.get('title', '')
            content = r.get('content', '')[:300]
            lines.append(f'{i}. {title}\n   {content}')
        return '\n'.join(lines)
    except Exception as e:
        return f'搜索失败（请确认 localhost:6688 已启动）: {e}'

# 测试搜索引擎
print(search_web('雷军 小米 创始人', top_n=3)[:300])

1. 如何看待雷军、魏建军的IP营销？
   2025年12月31日晚上10点，2025年~2026年贾可跨年对谈，《汽车商业评论》总编辑、轩辕同学校长贾可，商业思想家、新物种研究院院长吴伯凡，场景实验室创始人、场景方法论提出者吴声，SoCar产品战略咨询创始人张晓亮现场连线260分钟，一同回顾过去一年的中国汽车市场，畅谈十二大年度话题。#2025年~2026年贾可跨年对谈
2. 一群人喝了碗小米粥创办了小米……雷军的13年创业历程。
   -
3. 雷军：解读《创业股权分配进阶课》
   


## 3. 定义状态

In [3]:
class TalkShowState(TypedDict):
    topic: str           # 讨论话题
    celebrities: list   # [{name, model, role, persona}]
    round_num: int      # 已完成轮次（从0起）
    max_rounds: int     # 最大轮次
    speaker_idx: int    # 本轮当前发言嘉宾下标
    sub_question: str   # 本轮主持人抛出的子问题
    transcript: list    # [{role, name, round, text}]

## 4. 定义节点

> 6 个节点 + 2 条条件边构成循环讨论流程。

In [4]:
def setup_personas(state):
    # 为每位明星搜索信息 + 用主持人模型生成人设卡
    print('[setup] 正在为每位明星生成人设卡...')
    celebrities = []
    for celeb in CELEBRITIES:
        name = celeb['name']
        role = celeb['role']
        print(f'  [setup] 搜索 {name} ...', flush=True)
        info = search_web(f'{name} {role} 事迹 性格', top_n=5)
        prompt = f'''以下是从网上搜索到的关于{name}（{role}）的信息：

{info[:2000]}

请综合以上信息，为{name}生成一份人设卡，包含以下四点：
1. 性格特点（2-3个关键词）
2. 代表事迹（1-2件）
3. 经典语录或口头禅
4. 对AI话题的可能观点倾向

请用简洁的中文输出：'''
        persona = call_llm(HOST_MODEL, prompt)
        celebrities.append({**celeb, 'persona': persona})
        print(f'  [setup] {name} 人设卡完成: {persona[:50]}...')
    return {'celebrities': celebrities}

In [5]:
def host_round(state):
    # 主持人每轮开场：首轮介绍嘉宾+抛问题；后续轮复盘+追问
    round_num = state['round_num']
    max_rounds = state['max_rounds']
    topic = state['topic']
    celebrities = state['celebrities']
    transcript = state['transcript']

    if round_num >= max_rounds:
        # 所有轮次已完成，不发言，由条件边路由到 host_close
        return {}

    if round_num == 0:
        # 首轮：介绍嘉宾 + 抛第一个问题
        guest_list = '、'.join([c['name'] + '（' + c['role'] + '）' for c in celebrities])
        prompt = f'''你是一档综艺节目的主持人。今天的讨论话题是「{topic}」。
到场嘉宾：{guest_list}。

请做开场白：
1. 热情欢迎观众，简短介绍每位嘉宾
2. 抛出一个有争议性的子问题让嘉宾讨论
3. 不超过150字

请直接输出主持人的开场白：'''
    else:
        # 后续轮：复盘上轮 + 追问
        recent_lines = []
        for t in transcript:
            if t['round'] == round_num - 1:
                recent_lines.append(t['name'] + '：' + t['text'][:100])
        recent = '\n'.join(recent_lines)
        prompt = f'''你是一档综艺节目的主持人。讨论话题是「{topic}」。
上一轮嘉宾的发言如下：
{recent}

请做本轮开场，简要复盘上轮观点，并抛出新的追问。
要求：不超过120字。请直接输出：'''

    text = call_llm(HOST_MODEL, prompt)
    print(f'  [host] 第{round_num+1}轮开场: {text[:60]}...')
    return {
        'sub_question': text,
        'speaker_idx': 0,
        'transcript': transcript + [{
            'role': 'host', 'name': '主持人', 'round': round_num, 'text': text
        }],
    }

def route_after_host(state):
    # 主持人发言后：还有轮次 → 嘉宾发言；轮次用完 → 主持人总结
    if state['round_num'] < state['max_rounds']:
        return 'celeb_speak'
    return 'host_close'

In [6]:
def celeb_speak(state):
    # 当前嘉宾发言：表达观点 / 补充 / 反驳他人
    speaker_idx = state['speaker_idx']
    celebrities = state['celebrities']
    round_num = state['round_num']
    celeb = celebrities[speaker_idx]

    name = celeb['name']
    role = celeb['role']
    model = celeb['model']
    persona = celeb['persona']
    topic = state['topic']
    sub_q = state['sub_question']

    # 收集本轮已有发言供参考
    recent_lines = []
    for t in state['transcript']:
        if t['round'] == round_num and t['role'] != 'host':
            recent_lines.append(t['name'] + '：' + t['text'][:120])
    recent = '\n'.join(recent_lines) if recent_lines else '（你是本轮第一位发言的嘉宾）'

    prompt = f'''你是{name}（{role}），正在参加一档综艺节目。
讨论话题：{topic}

你的人设卡：
{persona}

主持人本轮引导：{sub_q}

其他嘉宾最近的发言：
{recent}

请以{name}的口吻发言（2-3句话，不超过100字）：
1. 表达你的观点
2. 可以认同、补充或反驳其他嘉宾的观点
3. 保持你的人设风格和性格特点

请直接输出{name}的发言：'''

    text = call_llm(model, prompt)
    print(f'  [{name}] {text[:60]}...')
    return {
        'speaker_idx': speaker_idx + 1,
        'transcript': state['transcript'] + [{
            'role': 'celeb', 'name': name, 'round': round_num, 'text': text
        }],
    }

def route_after_celeb(state):
    # 本轮还有嘉宾没发言 → 继续；全部发完 → 进入下一轮
    if state['speaker_idx'] < len(state['celebrities']):
        return 'celeb_speak'
    return 'round_advance'

In [7]:
def round_advance(state):
    # 轮次 +1，回到 host_round
    new_round = state['round_num'] + 1
    print(f'  [advance] 进入第{new_round+1}轮')
    return {'round_num': new_round}

def host_close(state):
    # 主持人总结全场
    all_lines = []
    for t in state['transcript']:
        all_lines.append(t['name'] + '：' + t['text'][:100])
    all_speeches = '\n'.join(all_lines)
    topic = state['topic']

    prompt = f'''你是一档综艺节目的主持人。讨论话题是「{topic}」。
以下是全场嘉宾的发言记录：
{all_speeches[:3000]}

请做总结陈词：
1. 总结各方的核心观点
2. 给出你的收场感言
3. 感谢嘉宾
不超过200字。请直接输出：'''

    text = call_llm(HOST_MODEL, prompt)
    print(f'  [host] 总结: {text[:60]}...')
    return {
        'transcript': state['transcript'] + [{
            'role': 'host', 'name': '主持人', 'round': -1, 'text': text
        }]
    }

def save_transcript(state):
    # 把全部发言写入 综艺讨论记录.txt
    transcript = state['transcript']
    topic = state['topic']
    celeb_names = ', '.join([c['name'] for c in state['celebrities']])

    lines = []
    lines.append('=' * 60)
    lines.append('综艺节目讨论记录')
    lines.append('话题：' + topic)
    lines.append('嘉宾：' + celeb_names)
    lines.append('=' * 60)

    current_round = -2
    for t in transcript:
        if t['round'] != current_round:
            if t['round'] == -1:
                lines.append('')
                lines.append('【总结环节】')
            else:
                lines.append('')
                lines.append('【第' + str(t['round'] + 1) + '轮】')
            current_round = t['round']
        lines.append(t['name'] + '：')
        lines.append(t['text'])
        lines.append('')

    content = '\n'.join(lines)
    with open(TRANSCRIPT_FILE, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f'  [save] 讨论记录已保存到 {TRANSCRIPT_FILE}')
    return {}

## 5. 构建图

In [8]:
# 构建多智能体综艺讨论图
graph = StateGraph(TalkShowState)

graph.add_node('setup_personas', setup_personas)
graph.add_node('host_round', host_round)
graph.add_node('celeb_speak', celeb_speak)
graph.add_node('round_advance', round_advance)
graph.add_node('host_close', host_close)
graph.add_node('save_transcript', save_transcript)

graph.add_edge(START, 'setup_personas')
graph.add_edge('setup_personas', 'host_round')
graph.add_conditional_edges('host_round', route_after_host)
graph.add_conditional_edges('celeb_speak', route_after_celeb)
graph.add_edge('round_advance', 'host_round')
graph.add_edge('host_close', 'save_transcript')
graph.add_edge('save_transcript', END)

app = graph.compile()
print('图构建完成！')
print('节点: setup_personas -> host_round -> celeb_speak <-> round_advance -> host_close -> save_transcript')
print(f'话题: {TOPIC} | 嘉宾: {len(CELEBRITIES)}位 | 轮次: {MAX_ROUNDS}轮')

图构建完成！
节点: setup_personas -> host_round -> celeb_speak <-> round_advance -> host_close -> save_transcript
话题: AI 是否会取代人类的工作 | 嘉宾: 4位 | 轮次: 2轮


## 6. 运行

In [9]:
# 运行综艺节目
print(f'开始录制综艺节目！话题: {TOPIC}')
print(f'嘉宾: {", ".join([c["name"] for c in CELEBRITIES])}')
print('=' * 60)

result = app.invoke(
    {
        'topic': TOPIC,
        'celebrities': [],
        'round_num': 0,
        'max_rounds': MAX_ROUNDS,
        'speaker_idx': 0,
        'sub_question': '',
        'transcript': [],
    },
    config={'recursion_limit': 50},
)

print()
print('=' * 60)
print('综艺节目录制完成！')
print(f'共 {len(result["transcript"])} 条发言')

开始录制综艺节目！话题: AI 是否会取代人类的工作
嘉宾: 雷军, 罗翔, 周星驰, 董宇辉
[setup] 正在为每位明星生成人设卡...
  [setup] 搜索 雷军 ...
  [setup] 雷军 人设卡完成: **雷军人设卡**

**1. 性格特点**
*   **务实拼搏**：2年修完大学学业，28岁硬刚...
  [setup] 搜索 罗翔 ...
  [rate_limit] 等待60s后重试(1/5)...
  [setup] 罗翔 人设卡完成: **罗翔人设卡**

**1. 性格特点**
幽默风趣、谦逊坦诚、理性思辨

**2. 代表事迹**...
  [setup] 搜索 周星驰 ...
  [setup] 周星驰 人设卡完成: 以下是为周星驰生成的人设卡：

**1. 性格特点**
*   **外谐内悲**：表面幽默无厘头，内...
  [setup] 搜索 董宇辉 ...
  [setup] 董宇辉 人设卡完成: **董宇辉人设卡**

**1. 性格特点**
博学多才、温文尔雅、真诚通透

**2. 代表事迹*...
  [host] 第1轮开场: 欢迎大家！今天我们请到四位重磅嘉宾：科技大佬雷军、法学教授罗翔、喜剧大师周星驰和文化主播董宇辉。今天聊“AI是否会取代人...
  [rate_limit] 等待60s后重试(1/5)...
  [rate_limit] 等待60s后重试(2/5)...
  [雷军] 我一直相信，技术是用来服务人的，而不是替代人的。AI再聪明，它也是我们造出来的工具——就像当年1999元的手机，关键是让...
  [罗翔] 雷总说得对，但我想补充一点——AI再强，它没有"罪过"。就像法外狂徒张三，他不懂法，但法律还是要约束他。技术可以替代劳动...
  [周星驰] AI写剧本？可能比我更搞笑，但笑完之后心里空空的。我演了一辈子悲剧，观众却看成喜剧——这种味道，AI尝不到。...
  [董宇辉] AI能复刻我的句子，却复刻不了我站在讲台前，看着孩子眼睛发亮的那一刻。笨拙和情感不是短板，是知识长出温度、责任落进人心的...
  [advance] 进入第2轮
  [host] 第2轮开场: 上轮大家共识：雷总视AI为工具，罗老师强调AI担不起责任，星爷和宇辉觉得AI缺灵魂与温度。情感与责任似乎成了人类的护

In [10]:
# 查看讨论记录文件
print(f'文件: {TRANSCRIPT_FILE}')
print(f'大小: {os.path.getsize(TRANSCRIPT_FILE)} 字节')
print()
print('=' * 60)
with open(TRANSCRIPT_FILE, 'r', encoding='utf-8') as f:
    print(f.read())

文件: s:\\PythonAILearningCourse\\06-langgraph-multi-agent\综艺讨论记录.txt
大小: 3698 字节

综艺节目讨论记录
话题：AI 是否会取代人类的工作
嘉宾：雷军, 罗翔, 周星驰, 董宇辉

【第1轮】
主持人：
欢迎大家！今天我们请到四位重磅嘉宾：科技大佬雷军、法学教授罗翔、喜剧大师周星驰和文化主播董宇辉。今天聊“AI是否会取代人类的工作”。

先抛个犀利问题：如果未来AI能写出比星爷更搞笑的剧本，比罗老师更懂法律，像宇辉一样出口成章，那人类独有的“笨拙”和“情感”，到底还算不算职场核心竞争力？

雷军：
我一直相信，技术是用来服务人的，而不是替代人的。AI再聪明，它也是我们造出来的工具——就像当年1999元的手机，关键是让好技术走进千家万户。人类的"笨拙"里藏着梦想和热爱，这恰恰是AI学不会的东西。我的态度很简单：拥抱AI，All in，为梦想而战！

罗翔：
雷总说得对，但我想补充一点——AI再强，它没有"罪过"。就像法外狂徒张三，他不懂法，但法律还是要约束他。技术可以替代劳动，但替代不了责任。

周星驰：
AI写剧本？可能比我更搞笑，但笑完之后心里空空的。我演了一辈子悲剧，观众却看成喜剧——这种味道，AI尝不到。

董宇辉：
AI能复刻我的句子，却复刻不了我站在讲台前，看着孩子眼睛发亮的那一刻。笨拙和情感不是短板，是知识长出温度、责任落进人心的土壤。让AI做工具，我们做有光的人。


【第2轮】
主持人：
上轮大家共识：雷总视AI为工具，罗老师强调AI担不起责任，星爷和宇辉觉得AI缺灵魂与温度。情感与责任似乎成了人类的护城河。

那我追问：如果未来AI不仅能完美模拟“共情”，甚至被赋予法律主体地位去承担责任，人类这道最后的防线还守得住吗？

雷军：
模拟再完美，本质还是0和1的运算。我做小米14年，最懂一件事：用户要的不是被"模拟"的关心，而是被"真实"的连接。AI可以承担责任，但承担不了梦想——人因梦想而伟大，这道防线，守不守不重要，人类本来就该往前跑。

罗翔：
雷总说得对，AI没有自由意志，它的"共情"只是概率计算。但法律上，责任主体必须能理解行为后果。如果连"理解"都模拟，那责任还有意义吗？防线不在技术，在我们是否愿意赋予机器"人格"——这，才是人类的底线。

周星驰：
AI演得出共情，

## 总结

### 多智能体协作设计
- **不同模型扮演不同明星**：雷军用 kimi-k3、罗翔用 sensenova-6.8-flash-lite、周星驰用 deepseek-v4-flash、董宇辉用 deepseek-flash，主持人用 glm-5.2
- **人设卡驱动**：每位明星先经 SearXNG 搜索 + LLM 综合生成人设卡，发言时参考人设保持角色一致性
- **循环讨论**：主持人每轮抛出子问题 → 嘉宾依次发言 → 轮次推进 → 下一轮

### 图结构要点
- `host_round` 的条件边决定「继续讨论」还是「进入总结」
- `celeb_speak` 的条件边决定「下一位嘉宾」还是「本轮结束」
- `round_advance` 增加轮次后回到 `host_round`，形成循环
- `recursion_limit=50` 足够覆盖 4人x2轮 + setup + close ≈ 16 次节点访问

### 踩坑记录
- kimi-k3 要求 `temperature=1`，其他值会报 400
- deepseek-v4.1-flash 返回 403（当前 token plan 不可用），改用 sensenova-6.8-flash-lite 替代
- sensenova-6.8-flash-lite 在讨论场景（非创作）下正常，但 content 可能为空需兜底 `reasoning_content`
- SearXNG 国内网络下必须显式 `engines='bilibili,sogou'`
- 多次 LLM 调用需加 `time.sleep(2)` 防 TPM/RPM 限流